In [1]:
"""
compare_moments.py
==================
Loads two simulated income CSVs (Python DGP and Fortran DGP),
computes the same moments for each using the toolbox_final logic,
and prints a side-by-side comparison to prove they share the same DGP.

No files are saved.

Usage:
    python compare_moments.py
"""

import numpy as np
import pandas as pd
from scipy import stats

# ============================================================================
# PATHS -- update these to point to your two CSV files
# ============================================================================
PATH_PY   = r'C:\Users\adv23\Dropbox\John-Jackson-Steve 2023\earning_dynamics\data\intermediate\simulated_guv_data.csv'
PATH_FORT = r'C:\Users\adv23\Dropbox\John-Jackson-Steve 2023\earning_dynamics\data\intermediate\simulated_guv_fortran.csv'   # output of simulate_guv.f90

# ============================================================================
# PARAMETERS (from toolbox_final.ipynb)
# ============================================================================
NSIM       = 50000
HMAX       = 36
NVASEINC   = 13
NVASEMNT   = 3
NIRINC     = 8
NIRCHG     = 10
NLAG       = 5
NLTINCPCT  = 15
LTH        = 8
EMPCDF_NUM = HMAX + 1
MINOBS     = 3
MINEMP     = 15
RMINWAGE   = 1.5
DPMISSING  = 1.0e15

VASEINCPCT  = np.array([1,2,11,21,31,41,51,61,71,81,91,96,100,101])
IRAVGINCPCT = np.array([1,6,11,31,51,71,91,96,101])
IRCHGPCT    = np.array([1,3,6,11,31,51,71,91,96,99,101])
LTINCPCT    = np.array([1,2,6,11,21,31,41,51,61,71,81,91,96,98,100,101])
NAGEBIN     = np.array([2,2,2])
AGEBINL     = np.array([1,9,19]) - 1
DF1         = np.array([2,6,1,2,3,4,6,11]) - 1
DF2         = np.array([1,1,0,0,0,0,0,0]) - 1

# ============================================================================
# UTILITY FUNCTIONS (from toolbox_final.ipynb)
# ============================================================================

def mean_var_miss(income):
    valid = income >= RMINWAGE
    if np.sum(valid) < 2:
        return 0.0, 0.0
    log_income = np.log(income[valid])
    return np.mean(log_income), np.var(log_income, ddof=1)

def mean_miss(x):
    valid = x < (DPMISSING - 1.0)
    if np.sum(valid) == 0:
        return 0.0
    return np.mean(x[valid])

def demean_col(x):
    valid = x < (DPMISSING - 1.0)
    if np.sum(valid) > 0:
        x[valid] = x[valid] - np.mean(x[valid])

def sdskewkurt_miss(x):
    valid = x < (DPMISSING - 1.0)
    if np.sum(valid) < 2:
        return 0.0, 0.0, 0.0
    x_valid = x[valid]
    sd   = np.std(x_valid, ddof=1)
    if sd == 0:
        return sd, 0.0, 0.0
    skew = stats.skew(x_valid, bias=False)
    kurt = stats.kurtosis(x_valid, bias=False)
    return sd, skew, kurt

def sortrows_col1(a):
    valid_mask = a[:, 0] < (DPMISSING - 1.0)
    valid_rows = a[valid_mask]
    invalid_rows = a[~valid_mask]
    if len(valid_rows) == 0:
        return a, 0
    sorted_idx = np.argsort(valid_rows[:, 0])
    result = np.vstack([valid_rows[sorted_idx], invalid_rows])
    return result, len(valid_rows)

# ============================================================================
# MAIN MOMENTS CALCULATION (from toolbox_final.ipynb)
# ============================================================================

def calculate_moments(ysim_in):
    nsim, hmax = ysim_in.shape

    SSK_L1 = np.zeros((3, NVASEINC, NVASEMNT))
    SSK_L5 = np.zeros((3, NVASEINC, NVASEMNT))
    irm    = np.zeros((2, NIRINC, NIRCHG, NLAG+1))
    incg   = np.zeros((NLTINCPCT, LTH))
    varlny = np.zeros(hmax)
    ecdf   = np.zeros(EMPCDF_NUM)

    agedum    = np.zeros(hmax)
    avgagedum = np.zeros(hmax)

    for h in range(hmax):
        agedum[h], varlny[h] = mean_var_miss(ysim_in[:, h])
    agedum = np.exp(agedum)

    for h in range(hmax):
        avgagedum[h] = np.mean(agedum[max(0, h-4):h+1])

    NLONG    = min(28, hmax - 2)
    longdata = np.full((nsim * NLONG, 9), DPMISSING)

    for h in range(2, min(30, hmax)):
        lb = (h - 2) * nsim
        ub = (h - 1) * nsim

        numobs     = np.zeros(nsim, dtype=int)
        avgpastinc = np.zeros(nsim)
        numobs[ysim_in[:, h] < RMINWAGE] = -5

        for j in range(min(h + 1, 5)):
            avgpastinc += np.maximum(ysim_in[:, h - j], RMINWAGE)
            numobs[ysim_in[:, h - j] >= RMINWAGE] += 1

        valid = numobs >= MINOBS
        avgpastinc[valid]  = avgpastinc[valid] / (min(h + 1, 5) * avgagedum[h])
        avgpastinc[~valid] = DPMISSING
        longdata[lb:ub, 0] = avgpastinc

        for j in range(8):
            h_fut  = h + DF1[j] + 1
            h_base = h + DF2[j] + 1
            if h_fut < hmax:
                y_base     = ysim_in[:, h_base] / agedum[h_base]
                y_fut      = ysim_in[:, h_fut]  / agedum[h_fut]
                valid_both = (ysim_in[:, h_base] >= RMINWAGE) | (ysim_in[:, h_fut] >= RMINWAGE)
                arc_chg    = np.full(nsim, DPMISSING)
                arc_chg[valid_both] = (
                    2.0 * (y_fut[valid_both] - y_base[valid_both]) /
                    (y_fut[valid_both] + y_base[valid_both])
                )
                longdata[lb:ub, 1 + j] = arc_chg

        for l in range(NLAG + 1):
            if h + DF1[2 + l] + 1 < hmax:
                demean_col(longdata[lb:ub, 3 + l])

    for i in range(3):
        for nh in range(NAGEBIN[i]):
            if i == 0:
                lb2, ub2 = (0, 3*nsim) if nh == 0 else (3*nsim, 8*nsim)
            else:
                hstart = AGEBINL[i] + 5 * nh
                lb2    = hstart * nsim
                ub2    = lb2 + 5 * nsim
            if ub2 > len(longdata):
                continue
            temp, nonmiss = sortrows_col1(longdata[lb2:ub2, 0:3].copy())
            for j in range(NVASEINC):
                lb3 = int(np.floor(nonmiss * (VASEINCPCT[j]   - 1) / 100))
                ub3 = min(int(np.floor(nonmiss * (VASEINCPCT[j+1] - 1) / 100)), nonmiss - 1)
                if ub3 > lb3:
                    ssk  = sdskewkurt_miss(temp[lb3:ub3+1, 1])
                    ssk5 = sdskewkurt_miss(temp[lb3:ub3+1, 2])
                    SSK_L1[i, j, :] += np.array(ssk)  / NAGEBIN[i]
                    SSK_L5[i, j, :] += np.array(ssk5) / NAGEBIN[i]

    for i in range(2):
        lb2 = 0          if i == 0 else 8  * nsim
        ub2 = 8 * nsim   if i == 0 else min(23 * nsim, len(longdata))
        if ub2 <= lb2:
            continue
        temp, nonmiss = sortrows_col1(np.column_stack([
            longdata[lb2:ub2, 0],
            longdata[lb2:ub2, 3:9]
        ]))
        for j in range(NIRINC):
            lb3 = int(np.floor(nonmiss * (IRAVGINCPCT[j]   - 1) / 100))
            ub3 = min(int(np.floor(nonmiss * (IRAVGINCPCT[j+1] - 1) / 100)), nonmiss - 1)
            if ub3 <= lb3:
                continue
            temp2, nonmiss2 = sortrows_col1(temp[lb3:ub3+1, 1:7].copy())
            for k in range(NIRCHG):
                lb4 = int(np.floor(nonmiss2 * (IRCHGPCT[k]   - 1) / 100))
                ub4 = min(int(np.floor(nonmiss2 * (IRCHGPCT[k+1] - 1) / 100)), nonmiss2 - 1)
                if ub4 > lb4:
                    for l in range(NLAG + 1):
                        irm[i, j, k, l] = mean_miss(temp2[lb4:ub4+1, l])

    emp   = np.zeros(nsim, dtype=int)
    LTinc = np.zeros((nsim, LTH + 1))
    jj    = 0
    for h in range(hmax):
        LTinc[:, 0] += np.maximum(ysim_in[:, h], RMINWAGE)
        emp[ysim_in[:, h] >= RMINWAGE] += 1
        if (h % 5 == 0) and (jj < LTH):
            LTinc[:, 1 + jj] = np.maximum(ysim_in[:, h], RMINWAGE)
            jj += 1

    LTinc[:, 0] = LTinc[:, 0] / hmax
    LTinc[emp < MINEMP, 0] = DPMISSING

    for i in range(EMPCDF_NUM - 1):
        ecdf[i] = 100.0 * np.sum(emp <= i) / nsim
    ecdf[-1] = 100.0

    temp_lt, nonmiss = sortrows_col1(LTinc.copy())
    for j in range(NLTINCPCT):
        lb2 = int(np.floor(nonmiss * (LTINCPCT[j]   - 1) / 100))
        ub2 = min(int(np.floor(nonmiss * (LTINCPCT[j+1] - 1) / 100)), nonmiss - 1)
        for h in range(LTH):
            if ub2 > lb2:
                incg[j, h] = mean_miss(temp_lt[lb2:ub2+1, h+1])

    return {
        'SdSkewKurt_L1': SSK_L1,
        'SdSkewKurt_L5': SSK_L5,
        'irmoments':     irm,
        'incgrwth':      incg,
        'var_lny':       varlny,
        'EmpCDF':        ecdf,
    }

# ============================================================================
# COMPARISON HELPERS
# ============================================================================

def print_section(title):
    print(f'\n{"="*70}')
    print(f'  {title}')
    print(f'{"="*70}')

def compare_1d(label, a, b, fmt='.4f'):
    """Print side-by-side 1D arrays and their max absolute difference."""
    print(f'\n--- {label} ---')
    print(f'{"Age/Idx":<8}  {"Python":>12}  {"Fortran":>12}  {"Abs Diff":>12}')
    print('-' * 50)
    for i, (x, y) in enumerate(zip(a, b)):
        print(f'{i+1:<8}  {x:>12{fmt}}  {y:>12{fmt}}  {abs(x-y):>12{fmt}}')
    print(f'\n  Max abs diff: {np.max(np.abs(a - b)):.6f}')
    print(f'  Mean abs diff: {np.mean(np.abs(a - b)):.6f}')

def compare_ssk(label, A, B):
    """Compare SdSkewKurt arrays [3, 13, 3]."""
    age_labels = ['Young', 'Middle', 'Old']
    mom_labels = ['SD', 'Skew', 'Kurt']
    print(f'\n--- {label} ---')
    for i, ag in enumerate(age_labels):
        for k, mom in enumerate(mom_labels):
            a = A[i, :, k]
            b = B[i, :, k]
            diff = np.abs(a - b)
            print(f'  {ag} / {mom:<5}  max_diff={np.max(diff):.4f}  '
                  f'mean_diff={np.mean(diff):.4f}  '
                  f'corr={np.corrcoef(a, b)[0,1]:.6f}')

def compare_ir(A, B):
    """Compare impulse response moments [2, 8, 10, 6]."""
    age_labels = ['Young', 'Old']
    print('\n--- Impulse Response Moments ---')
    for i, ag in enumerate(age_labels):
        a = A[i].flatten()
        b = B[i].flatten()
        diff = np.abs(a - b)
        print(f'  {ag:<8}  max_diff={np.max(diff):.4f}  '
              f'mean_diff={np.mean(diff):.4f}  '
              f'corr={np.corrcoef(a, b)[0,1]:.6f}')

def compare_incgrwth(A, B):
    """Compare lifetime income growth [15, 8]."""
    a = A.flatten()
    b = B.flatten()
    diff = np.abs(a - b)
    print(f'\n--- Lifetime Income Growth ---')
    print(f'  max_diff={np.max(diff):.4f}  '
          f'mean_diff={np.mean(diff):.4f}  '
          f'corr={np.corrcoef(a, b)[0,1]:.6f}')

# ============================================================================
# MAIN
# ============================================================================

if __name__ == '__main__':

    print('Loading Python CSV ...')
    df_py   = pd.read_csv(PATH_PY)
    ysim_py = df_py.values.astype(float)
    print(f'  Shape: {ysim_py.shape}')

    print('Loading Fortran CSV ...')
    df_fort   = pd.read_csv(PATH_FORT)
    ysim_fort = df_fort.values.astype(float)
    print(f'  Shape: {ysim_fort.shape}')

    print('\nCalculating moments for Python data ...')
    m_py = calculate_moments(ysim_py)

    print('Calculating moments for Fortran data ...')
    m_fort = calculate_moments(ysim_fort)

    # ── 1. Variance of log income by age ──────────────────────────────────
    print_section('1. Variance of Log Income by Age (var_lny)')
    compare_1d('var_lny', m_py['var_lny'], m_fort['var_lny'])

    # ── 2. Employment CDF ─────────────────────────────────────────────────
    print_section('2. Employment CDF')
    compare_1d('EmpCDF', m_py['EmpCDF'], m_fort['EmpCDF'], fmt='.3f')

    # ── 3. Cross-sectional moments (SdSkewKurt) ───────────────────────────
    print_section('3. Cross-Sectional Moments — 1-Year Changes (SdSkewKurt_L1)')
    compare_ssk('SdSkewKurt_L1', m_py['SdSkewKurt_L1'], m_fort['SdSkewKurt_L1'])

    print_section('4. Cross-Sectional Moments — 5-Year Changes (SdSkewKurt_L5)')
    compare_ssk('SdSkewKurt_L5', m_py['SdSkewKurt_L5'], m_fort['SdSkewKurt_L5'])

    # ── 4. Impulse response ───────────────────────────────────────────────
    print_section('5. Impulse Response Moments')
    compare_ir(m_py['irmoments'], m_fort['irmoments'])

    # ── 5. Lifetime income growth ─────────────────────────────────────────
    print_section('6. Lifetime Income Growth')
    compare_incgrwth(m_py['incgrwth'], m_fort['incgrwth'])

    # ── Summary ───────────────────────────────────────────────────────────
    print_section('SUMMARY — Max Absolute Difference Across All Moments')
    for key in ['SdSkewKurt_L1', 'SdSkewKurt_L5', 'irmoments', 'incgrwth', 'var_lny', 'EmpCDF']:
        a = m_py[key].flatten()
        b = m_fort[key].flatten()
        corr = np.corrcoef(a, b)[0, 1]
        print(f'  {key:<20}  max_diff={np.max(np.abs(a-b)):.4f}  '
              f'mean_diff={np.mean(np.abs(a-b)):.4f}  corr={corr:.6f}')

Loading Python CSV ...
  Shape: (50000, 36)
Loading Fortran CSV ...
  Shape: (50000, 36)

Calculating moments for Python data ...
Calculating moments for Fortran data ...

  1. Variance of Log Income by Age (var_lny)

--- var_lny ---
Age/Idx         Python       Fortran      Abs Diff
--------------------------------------------------
1               0.4634        0.4623        0.0011
2               0.4762        0.4753        0.0009
3               0.4903        0.4848        0.0055
4               0.5013        0.4996        0.0017
5               0.5138        0.5122        0.0017
6               0.5282        0.5225        0.0056
7               0.5466        0.5343        0.0123
8               0.5649        0.5507        0.0141
9               0.5816        0.5717        0.0099
10              0.5971        0.5864        0.0106
11              0.6093        0.6046        0.0047
12              0.6275        0.6243        0.0033
13              0.6532        0.6473        0.0059
1